In [8]:
import numpy as np
import pandas as pd


In [9]:
from tensorflow.keras import layers , models
from sklearn.utils.class_weight import compute_class_weight
import cv2 
from tensorflow.keras.applications import ResNet152V2

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

train_dir="skin-disease-datasaet/train_set"
test_dir="skin-disease-datasaet/test_set"

img_size = 224
batch_size = 32



data_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=False,
    brightness_range=[0.7, 1.3],
    channel_shift_range=20.0,
    fill_mode='nearest',
    featurewise_center=False,
    featurewise_std_normalization=False,
    preprocessing_function=preprocess_input,
    validation_split=0.25
)


train_gen = data_gen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    interpolation='bilinear',
    shuffle=True
)

valid_gen = data_gen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    interpolation='bilinear',
    shuffle=False
)

print("Class indices:", train_gen.class_indices)
print("Class sample counts:", np.bincount(train_gen.classes))

Found 695 images belonging to 8 classes.
Found 229 images belonging to 8 classes.
Class indices: {'BA- cellulitis': 0, 'BA-impetigo': 1, 'FU-athlete-foot': 2, 'FU-nail-fungus': 3, 'FU-ringworm': 4, 'PA-cutaneous-larva-migrans': 5, 'VI-chickenpox': 6, 'VI-shingles': 7}
Class sample counts: [102  60  93  97  68  75 102  98]


In [12]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.applications import ResNet50

In [13]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)
class_weights_dict = dict(enumerate(class_weights))
print("Class Weights:", class_weights_dict)

Class Weights: {0: 0.8517156862745098, 1: 1.4479166666666667, 2: 0.9341397849462365, 3: 0.895618556701031, 4: 1.2775735294117647, 5: 1.1583333333333334, 6: 0.8517156862745098, 7: 0.8864795918367347}


In [14]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam

base_model = ResNet152V2(include_top=False, input_shape=(img_size, img_size, 3))
base_model.trainable = False  

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(train_gen.num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 resnet152v2 (Functional)    (None, 7, 7, 2048)        58331648  
                                                                 
 global_average_pooling2d_1  (None, 2048)              0         
  (GlobalAveragePooling2D)                                       
                                                                 
 dense_2 (Dense)             (None, 256)               524544    
                                                                 
 dropout_1 (Dropout)         (None, 256)               0         
                                                                 
 dense_3 (Dense)             (None, 8)                 2056      
                                                                 
Total params: 58858248 (224.53 MB)
Trainable params: 526600 (2.01 MB)
Non-trainable params: 58331648 (222.52 MB)
_______

In [15]:
history=model.fit(
    train_gen,
    epochs=25,
    validation_data=valid_gen,
    class_weight=class_weights_dict
)

Epoch 1/25
22/22 [==============================] - 23s 774ms/step - loss: 1.4859 - accuracy: 0.5295 - val_loss: 0.7694 - val_accuracy: 0.7642
Epoch 2/25
22/22 [==============================] - 13s 605ms/step - loss: 0.6769 - accuracy: 0.7770 - val_loss: 0.6246 - val_accuracy: 0.7424
Epoch 3/25
22/22 [==============================] - 13s 611ms/step - loss: 0.4468 - accuracy: 0.8518 - val_loss: 0.5000 - val_accuracy: 0.8297
Epoch 4/25
22/22 [==============================] - 15s 699ms/step - loss: 0.4124 - accuracy: 0.8604 - val_loss: 0.4540 - val_accuracy: 0.8384
Epoch 5/25
22/22 [==============================] - 16s 750ms/step - loss: 0.3933 - accuracy: 0.8633 - val_loss: 0.6811 - val_accuracy: 0.8079
Epoch 6/25
22/22 [==============================] - 18s 814ms/step - loss: 0.3475 - accuracy: 0.8921 - val_loss: 0.3886 - val_accuracy: 0.8646
Epoch 7/25
22/22 [==============================] - 19s 851ms/step - loss: 0.2661 - accuracy: 0.8993 - val_loss: 0.3658 - val_accuracy: 0.8908

In [ ]:
model.save("ass3.h5")

/Users/ashishshirsath/tf-cnn-env/lib/python3.10/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
